<a href="https://colab.research.google.com/github/Hedil12/DnD_LLM/blob/main/500M_LLM_DnD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Downloading the needed Libraries
!pip install transformers trl datasets accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 530.9/530.9 kB 44.2 MB/s eta 0:00:00


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

model_id = model_id = "Qwen/Qwen2.5-0.5B-Instruct"

# 1. Download from API to RAM
print("Downloading model...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.float16
    )

# 2. Save to a permanent local folder
# This creates a folder you can just copy-paste to your offline CPU/Mobile
save_path = "./my_offline_base_model"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print(f"Model saved to {save_path}. You can now use this folder without internet.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/988M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to ./my_offline_base_model. You can now use this folder without internet.


In [ ]:
import requests
# issues the dataset cant be obtained easily due to a script present
# Datasets are in a folder where 1 data is 1 file aka split data

# obtain request from the dataset directory
api_url = "https://huggingface.co/api/datasets/lara-martin/FIREBALL/tree/main/filtered"
resp = requests.get(api_url).json()

# keep only .jsonl shards
shards = [f['path'] for f in resp if f['path'].endswith('.jsonl')]
print(f"found {len(shards)} shards")        # -> 100+ files

# build raw HTTPS urls
base = "https://huggingface.co/datasets/lara-martin/FIREBALL/resolve/main/"
urls = [base + s for s in shards]

# stream the dataset
from datasets import load_dataset
dataset = load_dataset("json", data_files={"train": urls})

In [ ]:
# --- Start of Proposed Change ---
# Inspect the dataset to understand its structure
print("Dataset features:", dataset["train"].features)
print("Sample data point:", dataset["train"][0])

# Define a formatting function to create the 'text' field
def formatting_func(example):
    # Pulling from your Fireball JSON structure
    history = " ".join(example.get("utterance_history", []))
    # This is what we WANT the model to output
    target_command = " ".join(example.get("commands_norm", []))

    # The 'Instruction' tells the model HOW to behave
    instruction = "Convert the D&D player intent into a valid Avrae bot command based on the combat state."

    # The 'Input' is the situation
    input_context = f"History: {history}"

    # The final string the model will train on
    text = (
        "### Instruction:\n" + instruction + "\n\n"
        "### Input:\n" + input_context + "\n\n"
        "### Response:\n" + target_command + "</s>"
    )
    return {"text": text}

In [ ]:
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

# Apply the mapping and formatting function to dataset
dataset["train"] = dataset["train"].map(formatting_func, remove_columns=dataset["train"].column_names)

training_args = SFTConfig(
    output_dir="./dnd_model_colab",
    max_steps=10000,              # Training for steps is safer than epochs on Colab
    per_device_train_batch_size=8,
    logging_steps=100,
    save_steps=2000,              # Saves checkpoints in case Colab crashes
    learning_rate=2e-5,
    packing=True,                 # Crucial for speed
    dataset_text_field="text",    # Ensure your mapping function outputs a 'text' field
    use_cpu=True,                 # Fix: Explicitly use CPU for training
)

trainer = SFTTrainer(
    model="./my_offline_base_model", # Load from your local save
    train_dataset=dataset["train"],
    args=training_args,
)

trainer.train()
# SAVE AGAIN: This is your final 'Smarter' model
trainer.save_model("./dnd_final_90m")

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# Push the model
trainer.push_to_hub("hedale/dnd-90m-model")

# Always push the tokenizer too, or the model won't know how to "read"
tokenizer.push_to_hub("hedale/dnd-90m-model")

In [ ]:
import torch
from transformers import pipeline

# Replace with your local model path
model_path = "./my_offline_base_model"

# Initialize the generator
# If you have a GPU, change device to 0. Otherwise, use -1 for CPU.
pipe = pipeline("text-generation",
                model=model_path,
                device=-1)

def run_test(context, player_message):
    # This structure mimics the formatting_func we talked about
    prompt = f"""### System: You are an Avrae D&D Assistant.
    Use the context to provide the correct !command.
    ### Combat Context:
    {context}

    ### Player Message:
    {player_message}

    ### Command:
    """
    # We set temperature to 0.1 to prevent "creativity" (hallucination)
    response = pipe(prompt, max_new_tokens=32, temperature=0.1, do_sample=False)
    print(response[0]['generated_text'])



def run_local_test(context, player_message, prompt):
    # We set temperature to 0.1 to prevent "creativity" (hallucination)
    response = pipe(prompt, max_new_tokens=32, temperature=0.1, do_sample=False)
    print(response[0]['generated_text'])

In [ ]:
from transformers import pipeline, set_seed

# Load model (Ensure you are using your base model for now)
generator = pipeline('text-generation', model='./my_offline_base_model', device=-1)

def test_dm_logic(situation, action):
    # We use a "Few-Shot" prompt here.
    # We show the model 2 examples of GOOD logic so it knows what to do.
    prompt = f"""
### System: You are a strict D&D 5e Dungeon Master. Resolve actions using game mechanics.

### Example 1:
Situation: A locked wooden door.
Action: I kick it open.
DM Logic: Wooden doors have AC 15 and 18 HP. Str check needed.
Response: Roll a Strength check to break the door down.

### Example 2:
Situation: A steep cliff.
Action: I climb up.
DM Logic: Climbing difficult surfaces requires a check.
Response: The rock is slippery. Roll Athletics to climb.

### Current Game:
Situation: {situation}
Action: {action}
DM Logic:"""

    # Fix: Remove max_length, use ONLY max_new_tokens
    result = generator(
        prompt,
        max_new_tokens=60,
        do_sample=True,
        temperature=0.3, # Low temp = More logical/boring (Less hallucination)
        pad_token_id=generator.tokenizer.eos_token_id
    )

    # We split by 'Response:' to get just the final answer
    print(result[0]['generated_text'])

# --- RUN THE TEST ---
test_dm_logic(
    "A massive boulder blocks the path.",
    "I use a pickaxe to break the boulder in half."
)

In [ ]:
dnd_context = "Actor: Nimue Willowleaf (HP 121/121). Target: Wanderer. Spells: Mage Armor, Shield, Misty Step."

test1 = "I cast Mage Armor on Nim"
test2 = "I'll shoot my crossbow at AS4"
test3 = "I'm bloodied, I need to use Second Wind"
testcase = [test1, test2, test3]

for i in testcase:
    run_local_test(dnd_context, i)
    print("\n\n")
print("test case complete")